In [30]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [31]:
# RNNScratch Prerequisite

class RNNScratch(d2l.Module):
    
    num_inputs: int
    num_hiddens: int
    sigma: float
    
    def __init__(
        self,
        num_inputs: int,
        num_hiddens: int,
        sigma: float = 1e-2,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()
        

        # [B, V] @ [V, H] -> [B, H]        
        self.W_xh = nn.Parameter(
            torch.randn(
                num_inputs,
                num_hiddens,
            ) * sigma
        )

        # [B, H] @ [H, H] -> [B, H]        
        self.W_hh = nn.Parameter(
            torch.randn(
                num_hiddens,
                num_hiddens,
            ) * sigma
        )
        
        # [H]
        self.b_h = nn.Parameter(
            torch.zeros(num_hiddens)
        )
        
        
def rnn_scratch_forward(
    self: RNNScratch,
    inputs: torch.Tensor,
    state: torch.Tensor | tuple[torch.Tensor] | None = None,
) -> tuple[list[torch.Tensor], torch.Tensor]:
    
    # inputs shape: X.T & One-hot encoding 적용한 상태
    # [T, B, V] = [num_steps, batch_size, num_inputs]
    if state is None:
        hidden_state = torch.zeros(
            (
                inputs.shape[1], # batch_size
                self.num_hiddens,
            ),
            device=inputs.device,
        )
    elif isinstance(state, tuple):
        hidden_state = state[0]
    else:
        hidden_state = state
        
        
    # 각 time step의 Hidden State를 저장
    outputs: list[torch.Tensor] = []
    
    # inputs의 첫 dimension이 time이므로 
    # X에는 한 time step의 minibatch가 들어온다. -> T만큼 반복
    for X in inputs:
        hidden_state = torch.tanh(
            X @ self.W_xh
            + hidden_state @ self.W_hh
            + self.b_h
        )

        outputs.append(hidden_state)
        
    # outputs: [T, B, H]
    return outputs, hidden_state
    
    
setattr(
    RNNScratch,
    "forward",
    rnn_scratch_forward,
)

In [32]:
# RNN-Based Language Model

class RNNLMScratch(d2l.Classifier):
    
    rnn: RNNScratch
    vocab_size: int
    lr: float

    def __init__(
        self,
        rnn: RNNScratch,
        vocab_size: int,
        lr: float = 1e-2,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()
        self.init_params()
        
        
    def init_params(self) -> None:
        
        # [B, H] @ [H, V] -> [B, V]
        self.W_hq = nn.Parameter(
            torch.randn(
                self.rnn.num_hiddens,
                self.vocab_size,
            ) * self.rnn.sigma
        )
        
        self.b_q = nn.Parameter(
            torch.zeros(
                self.vocab_size
            )
        )
        
        
    
    def training_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor],
    ) -> torch.Tensor:
        
        X, Y = batch

        loss = self.loss(
            self(X),
            Y,
        )
        
        # Perplexity = exp(Cross-Entropy)
        self.plot(
            "ppl",
            torch.exp(loss),
            train=True,
        )

        return loss
    
    def validation_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor],
    ) -> None:
        X, Y = batch

        loss = self.loss(
            self(X),
            Y,
        )

        self.plot(
            "ppl",
            torch.exp(loss),
            train=False,
        )

In [33]:
# One-Hot Encoding

token_indices = torch.tensor([0, 2])

encoded_tokens = F.one_hot(
    token_indices,
    num_classes=5,
)

print("encoded_tokens:")
print(encoded_tokens)



def rnn_lm_one_hot(
    self: RNNLMScratch,
    X: torch.Tensor,
) -> torch.Tensor:

    # X shape   :  [B, T]
    # X.T shape :  [T, B] 
    # One-Hot Encoding 이후: [T, B, V] 
    return F.one_hot(
        X.T,
        num_classes=self.vocab_size,
    ).to(dtype=torch.float32)
    

setattr(
    RNNLMScratch,
    "one_hot",
    rnn_lm_one_hot,
)

encoded_tokens:
tensor([[1, 0, 0, 0, 0],
        [0, 0, 1, 0, 0]])


In [34]:
# Hidden State to Vocabulary Logits

def rnn_lm_output_layer(
    self: RNNLMScratch,
    rnn_outputs: list[torch.Tensor],
) -> torch.Tensor:
    
    # rnn_outputs  : [T, B, H]
    # hidden_state : [B, H]
    #
    # Output Layer 적용 후:
    # [B, H] @ [H, V] + [V] -> [B, V]
    logits_per_step = [
        hidden_state @ self.W_hq + self.b_q
        for hidden_state in rnn_outputs
    ]
    
    # T개의 [B, V] Tensor를 합쳐 [B, T, V] 로 변환
    return torch.stack(
        logits_per_step,
        dim=1,
    )
    
    
def rnn_lm_forward(
    self: RNNLMScratch,
    X: torch.Tensor,
    state: torch.Tensor | tuple[torch.Tensor] | None = None,
) -> torch.Tensor:
    
    # X.T & One-Hot encoding:
    # [B, T] -> [T, B] -> [T, B, V]
    one_hot_inputs = self.one_hot(X)
    
    # 각 time step의 hidden state 계산
    # [T, B, V] -> [T, B, H], [B, H]
    rnn_outputs, _ = self.rnn(
        one_hot_inputs,
        state,
    )
    
    # 모든 hidden state를 vocabulary logit로 변환
    # [T, B, H] -> [B, T, V]
    return self.output_layer(
        rnn_outputs
    )
    
setattr(
    RNNLMScratch,
    "output_layer",
    rnn_lm_output_layer,
)

setattr(
    RNNLMScratch,
    "forward",
    rnn_lm_forward,
)

In [35]:
# Language Model Forward Check

batch_size = 2    # B
num_steps = 100   # T 
vocab_size = 16   # V = D
num_hiddens = 32  # H

rnn = RNNScratch(
    num_inputs=vocab_size,
    num_hiddens=num_hiddens,
)

model = RNNLMScratch(
    rnn=rnn,
    vocab_size=vocab_size,
)

In [36]:
# Language Model Shape & Loss Check

# [B, T]
X = torch.ones(
    (
        batch_size,
        num_steps,
    ),
    dtype=torch.int64,
)

# [B, T]
targets = torch.randint(
    low=0,
    high=vocab_size,
    size=(
        batch_size,
        num_steps,
    ),
)

one_hot_X = model.one_hot(X)
outputs = model(X)

loss = model.loss(
    outputs,
    targets,
)

perplexity = torch.exp(loss)

print(
    "Token indices: [B, T]",
    tuple(X.shape),
)
print(
    "One-hot inputs: [T, B, V]",
    tuple(one_hot_X.shape),
)
print(
    "Vocabulary logits: [B, T, V]",
    tuple(outputs.shape),
)
print(
    "Targets: [B, T]",
    tuple(targets.shape),
)
print(
    "Loss:",
    loss.detach().item(),
)
print(
    "Perplexity:",
    perplexity.detach().item(),
)


Token indices: [B, T] (2, 100)
One-hot inputs: [T, B, V] (100, 2, 16)
Vocabulary logits: [B, T, V] (2, 100, 16)
Targets: [B, T] (2, 100)
Loss: 2.7725934982299805
Perplexity: 16.000076293945312
